# Web Scraping

In [ ]:
#| include: false

import warnings
warnings.filterwarnings('ignore')

"Data is the new oil. It's valuable, but if unrefined it cannot really be used." -- Clive Humby, British mathematician

"The goal is to turn data into information, and information into insight." -- Carly Fiorina

"You can have data without information, but you cannot have information without data." -- Daniel Keys Moran

## Python Modules

We will use the following Python modules in this chapter that we have used in previous chapters:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

We will also use the following modules in this chapter:

In [ ]:
import requests
from bs4 import BeautifulSoup
from io import StringIO
import time

If you do not remember how to install new modules in Python, refer to Chapter 2 where there is a detailed discussion. The `requests` module is Python's most popular HTTP library for making web requests. `BeautifulSoup` (from the `beautifulsoup4` package) is an HTML and XML parsing library that makes it easy to extract data from web pages. We also import `StringIO` to help convert HTML strings into pandas DataFrames, and `time` to add polite pauses between requests.

In [ ]:
#| echo: false
#| eval: true

sns.set_theme(style="whitegrid")

## Getting Sports Data

Throughout this book, we use a variety of data sources. In the baseball chapter, we used the `pybaseball` module to access MLB statistics and Statcast data. In the football chapter, we used `nfl_data_py` to load NFL play-by-play data. The `isapy_data` module bundled with this book provides pre-cleaned datasets for many of the examples in these chapters.

However, not all sports data is available through a convenient Python package or API. Many valuable data sources exist only as tables on web pages, and extracting that data requires a different skill: **web scraping**. Web scraping is the process of programmatically extracting data from websites. It is a foundational skill for any data analyst, and it is especially important in sports analytics where much of the historical data lives on reference websites like [Hockey-Reference](https://www.hockey-reference.com), [Basketball-Reference](https://www.basketball-reference.com), and [Pro-Football-Reference](https://www.pro-football-reference.com).

In this chapter, we will learn how to scrape data from the web using Python. We will focus on extracting NHL hockey data from Hockey-Reference, but the techniques you learn here apply to any website.

## Web Scraping and Ethics

Before we write any code, it is important to understand the ethical and legal considerations around web scraping. Just because data is publicly visible on a website does not mean you have unrestricted rights to scrape it.

**Legal considerations.** The legality of web scraping depends on the jurisdiction and the specific circumstances. In the United States, the landmark case *hiQ Labs v. LinkedIn* (2022) established that scraping publicly available data does not violate the Computer Fraud and Abuse Act. However, this ruling has nuances, and scraping data that is behind a login wall, subject to a Terms of Service prohibition, or protected by copyright may still create legal risk. When in doubt, check the website's Terms of Service.

**The `robots.txt` file.** Most websites publish a file called `robots.txt` at their root URL that specifies which parts of the site automated tools (including scrapers) are allowed to access. This file is not legally binding, but it is considered a standard of good faith. You should always check a site's `robots.txt` before scraping.

In [ ]:
resp = requests.get("https://www.hockey-reference.com/robots.txt")
print(resp.text[:500])

**Best practices for ethical scraping:**

- **Rate limit your requests.** Add a delay between requests using `time.sleep()`. A minimum of 3 seconds between requests is a good rule of thumb. Never send requests as fast as your computer can.
- **Identify yourself.** Set a custom `User-Agent` header so the website operator can contact you if there is a problem.
- **Cache your results.** Save scraped data locally so you do not need to re-scrape the same pages. In this chapter, we use Quarto's `#| cache: true` directive to avoid re-scraping on every render.
- **Do not overload servers.** If you are scraping a large number of pages, spread your requests over time.
- **Attribute your sources.** Always credit the website from which you obtained your data.

## Web Document Structure

To scrape data from a web page, you need a basic understanding of how web documents are structured. Web pages are written in **HTML (HyperText Markup Language)**, which uses **tags** to define the structure and content of a page.

Here are the most important HTML elements for web scraping:

- `<table>`: Defines a data table. Tables contain rows (`<tr>`), header cells (`<th>`), and data cells (`<td>`).
- `<a href="...">`: A hyperlink. The `href` attribute contains the URL.
- `<div id="...">` and `<span class="...">`: Container elements used to organize page content. The `id` and `class` attributes are used to identify specific elements.

For example, here is a simplified version of the roster table on a Hockey-Reference team page:

```html
<table id="roster">
  <thead>
    <tr><th>No.</th><th>Player</th><th>Pos</th></tr>
  </thead>
  <tbody>
    <tr>
      <td>29</td>
      <td><a href="/players/m/mackina01.html">Nathan MacKinnon</a></td>
      <td>C</td>
    </tr>
    <tr>
      <td>8</td>
      <td><a href="/players/m/makMDca01.html">Cale Makar</a></td>
      <td>D</td>
    </tr>
  </tbody>
</table>
```

Notice that the table has an `id` attribute (`"roster"`), which we can use to find it on the page. Each player's name is wrapped in an `<a>` tag with an `href` attribute pointing to their individual stats page.

To explore the structure of any web page, you can use the **Inspect Element** feature in your browser. In Chrome or Firefox, right-click on any element and select "Inspect" to open the developer tools, which show you the underlying HTML. This is an indispensable tool when writing scraping code.

There are two common ways to select elements from an HTML document: **CSS selectors** and **XPath expressions**. CSS selectors (e.g., `#roster a` to find all links inside the element with `id="roster"`) are generally simpler and more Pythonic when using BeautifulSoup. XPath is more powerful but more verbose. In this chapter, we will use BeautifulSoup's built-in methods, which use a CSS-selector-like approach.

## Web Scraping in Python

The standard Python workflow for web scraping has four steps:

1. **Fetch** the web page using `requests.get()`.
2. **Parse** the HTML content using `BeautifulSoup`.
3. **Find** the elements you want using methods like `.find()` and `.find_all()`.
4. **Extract** the data from those elements (text, attributes, table contents).

This workflow is analogous to the R approach using `rvest` and `xml2`, where `read_html()` fetches and parses the page, `html_nodes()` finds elements, and `html_table()` or `html_text()` extracts data.

Let us start with a simple example. We will fetch the Colorado Avalanche's 2024-25 team page from Hockey-Reference and check that the request was successful.

In [ ]:
url = "https://www.hockey-reference.com/teams/COL/2025.html"
response = requests.get(url)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.text, 'lxml')
print(f"Page title: {soup.title.string}")

A status code of `200` means the request was successful. Other common codes include `404` (page not found), `403` (forbidden), and `429` (too many requests --- you are being rate limited).

## Scraping NHL Data

Now let us scrape some real data. We will extract the Colorado Avalanche roster and player statistics from Hockey-Reference.

### Extracting Tables {.unnumbered}

The quickest way to grab all tables from a web page is `pd.read_html()`, which is pandas' built-in HTML table reader.

In [ ]:
tables = pd.read_html(url)
print(f"Number of tables found: {len(tables)}")

Hockey-Reference pages contain many tables. We can identify the one we want --- the roster --- by using BeautifulSoup to find the table by its `id` attribute, then converting it to a DataFrame. This is more robust than relying on the table index, which may change if Hockey-Reference updates their page layout.

In [ ]:
roster_table = soup.find('table', {'id': 'roster'})
roster = pd.read_html(StringIO(str(roster_table)))[0]
roster.head()

### Extracting Player Links {.unnumbered}

The `pd.read_html()` function extracts table data but **not** hyperlinks. To get the URLs for each player's individual stats page, we need BeautifulSoup.

In R, the equivalent code uses XPath: `//*[(@id = "roster")]//a`. In Python with BeautifulSoup, we use the `.find_all()` method to locate the `<td>` cells that contain player names, then extract the `<a>` tag's `href` attribute from each.

In [ ]:
player_links = []
for row in roster_table.find_all('td', {'data-stat': 'player'}):
    a_tag = row.find('a')
    if a_tag:
        player_links.append(a_tag['href'])

print(f"Found {len(player_links)} player links")
print(player_links[:5])

### Scraping a Player's Stats Page {.unnumbered}

Let us scrape the career statistics for Nathan MacKinnon. We find his link in our list, construct the full URL, and fetch his page.

In [ ]:
mac_link = [l for l in player_links if 'mackina' in l][0]
player_url = f"https://www.hockey-reference.com{mac_link}"
print(f"Fetching: {player_url}")

time.sleep(3)  # Polite delay before request

player_resp = requests.get(player_url)
player_soup = BeautifulSoup(player_resp.text, 'lxml')

Hockey-Reference player pages contain several tables. We want the regular season stats table, which has the `id` of `"player_stats"`.

In [ ]:
stats_table = player_soup.find('table', {'id': 'player_stats'})
mac_stats = pd.read_html(StringIO(str(stats_table)))[0]

# The table has multi-level column headers. Flatten them by keeping
# the most informative level.
if isinstance(mac_stats.columns, pd.MultiIndex):
    mac_stats.columns = [col[1] if 'Unnamed' in str(col[0]) else col[1]
                         for col in mac_stats.columns]

# Clean column names (Python equivalent of R's janitor::clean_names())
mac_stats.columns = (mac_stats.columns
                     .str.lower()
                     .str.replace(' ', '_')
                     .str.replace('%', 'pct')
                     .str.replace('+/-', 'plus_minus'))

# Filter out summary rows
mac_stats = mac_stats[~mac_stats['season'].isin(['Career', '82 Game Avg.'])]
mac_stats.head()

## Writing a Custom Function

When you find yourself repeating the same scraping logic for different teams or players, it is time to wrap that logic into a reusable function. This makes your code cleaner, less error-prone, and easier to extend.

### Getting a Team Roster with Player Links {.unnumbered}

Our first function combines the roster table extraction and player link extraction into a single call. It takes a team abbreviation and season year, and returns a DataFrame with an additional `player_link` column.

In [ ]:
def get_hrc_player_links(team, year=2025):
    """Scrape roster with player links from hockey-reference.com.

    Parameters
    ----------
    team : str
        Three-letter team abbreviation (e.g., 'COL', 'DAL').
    year : int
        Season ending year (e.g., 2025 for the 2024-25 season).

    Returns
    -------
    pd.DataFrame
        Roster DataFrame with an additional 'player_link' column.
    """
    url = f"https://www.hockey-reference.com/teams/{team}/{year}.html"
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'lxml')

    # Get roster table
    roster_tag = soup.find('table', {'id': 'roster'})
    roster = pd.read_html(StringIO(str(roster_tag)))[0]

    # Get player links
    links = []
    for row in roster_tag.find_all('td', {'data-stat': 'player'}):
        a_tag = row.find('a')
        links.append(a_tag['href'] if a_tag else None)

    roster['player_link'] = links
    return roster

In [ ]:
col_roster = get_hrc_player_links("COL", year=2025)
col_roster[['No.', 'Player', 'Pos', 'Age', 'player_link']].head()

### A Simpler Roster Function {.unnumbered}

If you only need the roster table (without player links), a simpler function will do.

In [ ]:
def get_nhl_roster(team, season=2025):
    """Scrape team roster from hockey-reference.com.

    Parameters
    ----------
    team : str
        Three-letter team abbreviation.
    season : int
        Season ending year.

    Returns
    -------
    pd.DataFrame
        Roster table.
    """
    url = f"https://www.hockey-reference.com/teams/{team}/{season}.html"
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'lxml')
    roster_tag = soup.find('table', {'id': 'roster'})
    return pd.read_html(StringIO(str(roster_tag)))[0]

In [ ]:
time.sleep(3)
col_df = get_nhl_roster(team="COL", season=2025)
col_df.head()

### Looping Over Multiple Teams and Years {.unnumbered}

One of the most powerful applications of web scraping is collecting data across many teams and seasons. We can use nested loops to scrape multiple pages, building up a combined DataFrame.

In [ ]:
teams = ["COL", "DAL"]
years = [2024, 2025]
results = []

for team in teams:
    for year in years:
        print(f"Scraping {team} {year}...")
        roster = get_nhl_roster(team, year)
        roster['team'] = team
        roster['year'] = year
        results.append(roster)
        time.sleep(3)  # Polite delay between requests

results_df = pd.concat(results, ignore_index=True)
print(f"\nTotal rows: {len(results_df)}")
results_df.head()

Notice that we add `team` and `year` columns before concatenating, since the source tables do not contain that metadata. We also call `time.sleep(3)` between each request to avoid overwhelming the server.

## Comparing Scraped Data with Bundled Data

The `isapy_data` module provides pre-scraped hockey data so that you do not need to hit Hockey-Reference every time you work through this chapter. Let us compare our scraped data with the bundled datasets.

In [ ]:
import isapy_data

avs_roster = isapy_data.load_data("avs_roster_2022")
avs_stats = isapy_data.load_data("avs_stats_2022")

print(f"Bundled roster shape: {avs_roster.shape}")
print(f"Bundled stats shape: {avs_stats.shape}")
avs_roster.head()

The bundled data comes from the NHL's official API rather than Hockey-Reference, so the column names and formats differ. For example, the bundled roster uses `person_full_name` instead of `Player`, and `position_code` instead of `Pos`. When working with scraped data, you will often need to clean and standardize column names before joining or comparing with other data sources.

In [ ]:
#| label: fig-avs-goals
#| fig-cap: "Goals scored by Colorado Avalanche players in the 2021-22 season."
#| fig-width: 7

avs_2022 = avs_stats[avs_stats['season'] == 20212022].copy()
avs_2022 = avs_2022.sort_values('stat_goals', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(avs_2022['person_full_name'], avs_2022['stat_goals'])
ax.set_xlabel('Goals')
ax.set_ylabel('Player')
ax.set_title('Top 10 Goal Scorers: Colorado Avalanche, 2021-22')
plt.tight_layout()
plt.show()

## Conclusion

In this chapter, we learned the fundamentals of web scraping using Python. The `requests` library fetches web pages, and `BeautifulSoup` parses the HTML so we can extract the data we need.

We applied these skills to scrape NHL roster and player data from Hockey-Reference, building reusable functions and looping over multiple teams and seasons. Along the way, we emphasized the importance of ethical scraping practices: checking `robots.txt`, rate limiting with `time.sleep()`, and caching results to avoid unnecessary requests.

Web scraping is a foundational skill that you will use throughout the rest of this book and in your own sports analytics projects. Not all data comes neatly packaged in a Python module, and the ability to extract data from the web gives you access to a much broader universe of information.

___

::: {.callout-note icon=false}
## Lab Three: Web Scraping Hockey Data

In this lab, you will practice and extend the web scraping techniques from this chapter.

**Exercise 1.** Modify the `get_hrc_player_links()` function to also include each player's position (`Pos`) and age (`Age`) in the output. Call your modified function on the Dallas Stars (`"DAL"`) for the 2025 season.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

def get_hrc_player_links_v2(team, year=2025):
    """Enhanced roster scraper with position and age."""
    url = f"https://www.hockey-reference.com/teams/{team}/{year}.html"
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'lxml')
    roster_tag = soup.find('table', {'id': 'roster'})
    roster = pd.read_html(StringIO(str(roster_tag)))[0]

    links = []
    for row in roster_tag.find_all('td', {'data-stat': 'player'}):
        a_tag = row.find('a')
        links.append(a_tag['href'] if a_tag else None)
    roster['player_link'] = links
    return roster[['No.', 'Player', 'Pos', 'Age', 'player_link']]

dal_roster = get_hrc_player_links_v2("DAL", 2025)
dal_roster.head()

**Exercise 2.** Write a function `get_player_stats(player_link)` that takes a Hockey-Reference player link (e.g., `"/players/m/mackina01.html"`) and returns their career stats table as a clean DataFrame. Test it on a player of your choice. Remember to use `time.sleep(3)` before each request.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

def get_player_stats(player_link):
    """Scrape a player's career stats from hockey-reference.com."""
    url = f"https://www.hockey-reference.com{player_link}"
    time.sleep(3)
    resp = requests.get(url)
    soup = BeautifulSoup(resp.text, 'lxml')

    stats_table = soup.find('table', {'id': 'player_stats'})
    df = pd.read_html(StringIO(str(stats_table)))[0]

    # Flatten multi-level column headers
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[1] if 'Unnamed' in str(col[0]) else col[1]
                      for col in df.columns]

    df.columns = (df.columns.str.lower()
                  .str.replace(' ', '_')
                  .str.replace('%', 'pct'))
    df = df[~df['season'].isin(['Career', '82 Game Avg.'])]
    return df

mac_career = get_player_stats("/players/m/mackina01.html")
mac_career.head()

**Exercise 3.** Use the loop pattern from this chapter to scrape roster data for three teams of your choosing across the 2023, 2024, and 2025 seasons (9 total pages). Compute the average age of each team-season combination and create a grouped bar chart of the results. Remember to use `time.sleep(3)` between requests.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

lab_teams = ["COL", "DAL", "MIN"]
lab_years = [2023, 2024, 2025]
lab_results = []

for team in lab_teams:
    for year in lab_years:
        roster = get_nhl_roster(team, year)
        avg_age = roster['Age'].mean()
        lab_results.append({'team': team, 'year': year, 'avg_age': avg_age})
        time.sleep(3)

lab_df = pd.DataFrame(lab_results)

fig, ax = plt.subplots(figsize=(10, 6))
lab_pivot = lab_df.pivot(index='team', columns='year', values='avg_age')
lab_pivot.plot(kind='bar', ax=ax)
ax.set_xlabel('Team')
ax.set_ylabel('Average Age')
ax.set_title('Average Roster Age by Team and Season')
ax.legend(title='Season')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

:::

___

## Exercises

1. Use `requests` and `BeautifulSoup` to fetch the Colorado Avalanche 2025 page at `https://www.hockey-reference.com/teams/COL/2025.html`. How many HTML tables does the page contain? What are their `id` attributes?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

url = "https://www.hockey-reference.com/teams/COL/2025.html"
resp = requests.get(url)
soup = BeautifulSoup(resp.text, 'lxml')
tables = soup.find_all('table')

for t in tables:
    print(f"ID: {t.get('id', 'None')}")
print(f"\nTotal tables: {len(tables)}")

2. Scrape the Dallas Stars 2025 roster and clean the column names to be lowercase with underscores. Display the first 5 rows.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

url = "https://www.hockey-reference.com/teams/DAL/2025.html"
resp = requests.get(url)
soup = BeautifulSoup(resp.text, 'lxml')
roster_tag = soup.find('table', {'id': 'roster'})
roster = pd.read_html(StringIO(str(roster_tag)))[0]
roster.columns = (roster.columns.str.lower()
                  .str.replace('.', '', regex=False)
                  .str.replace(' ', '_')
                  .str.replace('/', '_'))
roster.head()

3. Using the `nhl_data_hockey_reference` dataset from `isapy_data`, filter to only Colorado Avalanche players (team abbreviation `"COL"`). How many unique players have played for the Avalanche in this dataset? What is the most common position?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

nhl_data = isapy_data.load_data("nhl_data_hockey_reference")
avs = nhl_data[nhl_data['tm'] == 'COL']
print(f"Unique Avalanche players: {avs['player'].nunique()}")
print(f"\nMost common position:")
print(avs.drop_duplicates('player')['pos'].value_counts().head())

4. Write a function that accepts a team abbreviation and returns the total number of goals scored across all seasons in the `nhl_data_hockey_reference` dataset. Test it with `"COL"` and `"DET"`.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

def team_total_goals(team_abbr):
    nhl_data = isapy_data.load_data("nhl_data_hockey_reference")
    team_data = nhl_data[nhl_data['tm'] == team_abbr]
    return team_data['g'].sum()

print(f"COL total goals: {team_total_goals('COL')}")
print(f"DET total goals: {team_total_goals('DET')}")

5. Using the `avs_stats_2022` dataset, create a scatter plot of goals (`stat_goals`) vs. assists (`stat_assists`) for all Avalanche players in the 2021-22 season (`season == 20212022`). Annotate any player with more than 20 goals with their name.

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

avs_stats = isapy_data.load_data("avs_stats_2022")
season_data = avs_stats[avs_stats['season'] == 20212022]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(season_data['stat_goals'], season_data['stat_assists'], alpha=0.7)
for _, row in season_data[season_data['stat_goals'] > 20].iterrows():
    ax.annotate(row['person_full_name'],
                (row['stat_goals'], row['stat_assists']),
                fontsize=8, alpha=0.8)
ax.set_xlabel('Goals')
ax.set_ylabel('Assists')
ax.set_title('Colorado Avalanche: Goals vs Assists, 2021-22')
plt.tight_layout()
plt.show()

6. Scrape the Colorado Avalanche 2025 roster using `get_nhl_roster()` and join it with the `nhl_data_hockey_reference` dataset by player name. How many players matched? What are common reasons for mismatches?

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

roster = get_nhl_roster("COL", 2025)
nhl_data = isapy_data.load_data("nhl_data_hockey_reference")
avs_hr = nhl_data[(nhl_data['tm'] == 'COL') & (nhl_data['season'] == 2025)]

merged = roster.merge(avs_hr, left_on='Player', right_on='player', how='left')
matched = merged['player'].notna().sum()
unmatched = merged[merged['player'].isna()]['Player'].tolist()
print(f"Matched: {matched}")
print(f"Unmatched: {len(unmatched)}")
print(f"Unmatched players: {unmatched[:10]}")
# Common reasons: name formatting differences, accented characters,
# players who appeared on roster but did not play in regular season

7. Using the loop pattern from this chapter, scrape roster data for the Central Division teams (`CHI`, `COL`, `DAL`, `MIN`, `NSH`, `STL`, `UTA`, `WPG`) for the 2025 season. Create a grouped bar chart showing the number of players at each position (F, D, G) for each team. You will need to map specific positions (C, LW, RW) to the broader category of forward (F).

In [ ]:
#| eval: false
#| code-fold: true
#| code-summary: "Show solution"

central_teams = ["CHI", "COL", "DAL", "MIN", "NSH", "STL", "UTA", "WPG"]
all_rosters = []
for team in central_teams:
    roster = get_nhl_roster(team, 2025)
    roster['team'] = team
    all_rosters.append(roster)
    time.sleep(3)

central_df = pd.concat(all_rosters, ignore_index=True)

# Map positions to broader categories
pos_map = {'C': 'F', 'LW': 'F', 'RW': 'F', 'D': 'D', 'G': 'G'}
central_df['pos_group'] = central_df['Pos'].map(pos_map)

pos_counts = central_df.groupby(['team', 'pos_group']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
pos_counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Team')
ax.set_ylabel('Number of Players')
ax.set_title('Roster Composition: NHL Central Division, 2024-25')
ax.legend(title='Position')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()